In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import time


from utils_ML import score_dataset
from utils_preproces import ols_fit, y_distribution_plot, corr_heatmap_plot, make_mi_scores, preprocess_data

from sklearn.ensemble import RandomForestRegressor, VotingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor, plot_importance
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold


from sklearn.metrics import mean_absolute_error

In [17]:
# Set Matplotlib defaults
plt.style.use("seaborn-v0_8-white")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

In [18]:
# Fix random state
random_state = 42

In [19]:
harps = pd.read_csv('harps_sample.csv')
apogee = pd.read_csv('apogee_sample.csv')
galah = pd.read_csv('galah_sample.csv')
gr8 = pd.read_csv('gr8_not_in_harps_sample.csv')

print (f'harps contains {len(harps)} stars')
print (f'apogee contains {len(apogee)} stars')    
print (f'galah contains {len(galah)} stars')
print (f'gr8 contains {len(gr8)} stars')

harps contains 798 stars
apogee contains 108526 stars
galah contains 116050 stars
gr8 contains 1436 stars


In [20]:
apogee = apogee[apogee.parallax >2]
print (len(apogee))
galah = galah[galah.parallax > 3]
print (len(galah))


23511
19678


# Here we focus on getting Fe from no abundance

In [21]:
columns_for_Fe = ['radial_velocity', 'V_lsr',  'U_lsr',  'W_lsr', 'P_gal', # 'parallax',  #'pmra', 'pmdec', #'ra', 'dec',
                'fe_h', ]

apogee = apogee[columns_for_Fe]
harps = harps[columns_for_Fe]
galah = galah[columns_for_Fe]
gr8 = gr8[columns_for_Fe]

### Identifying and exluding null values

In [22]:
apogee.isnull().sum() # 196 null value for P_gal
apogee.dropna(inplace = True)

In [23]:
galah.isnull().sum() # 1 null value for P_gal
galah.dropna(inplace = True)

In [24]:
#gr8.isnull().sum() # No null values

In [25]:
#harps.isnull().sum() # No null values

### Spliting data into Trianing and Testing sets

In [26]:
# Split APOGEE
apogee_train_full, apogee_test = train_test_split(apogee, test_size=0.1, random_state=random_state)
# Split GALAH
galah_train_full, galah_test = train_test_split(galah, test_size=0.3, random_state=random_state)
# Combine APOGEE and GALAH train sets
combined_train = pd.concat([apogee_train_full, galah_train_full, gr8], ignore_index=True)


In [ ]:
""" 
import h2o
from h2o.automl import H2OAutoML

h2o.init()

train_h2o = h2o.H2OFrame(combined_train)
test_h2o = h2o.H2OFrame(harps)

y_col = 'fe_h'  
x_cols = [col for col in train_h2o.columns if col != y_col]

aml = H2OAutoML(max_runtime_secs=600, seed=random_state)
aml.train(x=x_cols, y=y_col, training_frame=train_h2o)

pred = aml.leader.predict(test_h2o).as_data_frame().values.flatten()
print("H2O AutoML MAE:", mean_absolute_error(harps['fe_h'], pred))
"""


Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,23 mins 51 secs
H2O_cluster_timezone:,Europe/London
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,2 months and 3 days
H2O_cluster_name:,H2O_from_python_adbva_9yxbq2
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,6.198 Gb
H2O_cluster_total_cores:,22
H2O_cluster_allowed_cores:,22
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |█
22:47:31.893: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


c:\Users\adbva\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


H2O AutoML MAE: 0.17151231146030113


In [ ]:
# Show the leaderboard
#lb = aml.leaderboard
#leaderboard_df = lb.as_data_frame()
#leaderboard_df


c:\Users\adbva\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,StackedEnsemble_AllModels_3_AutoML_2_20250530_224731,0.180253,0.032491,0.139222,NaN,0.032491
1,StackedEnsemble_BestOfFamily_4_AutoML_2_20250530_224731,0.180295,0.032506,0.139272,NaN,0.032506
2,StackedEnsemble_AllModels_2_AutoML_2_20250530_224731,0.180399,0.032544,0.139327,NaN,0.032544
3,StackedEnsemble_AllModels_1_AutoML_2_20250530_224731,0.180473,0.032571,0.139345,NaN,0.032571
4,GBM_grid_1_AutoML_2_20250530_224731_model_6,0.180493,0.032578,0.139329,NaN,0.032578
5,GBM_grid_1_AutoML_2_20250530_224731_model_14,0.180500,0.032580,0.139441,NaN,0.032580
6,StackedEnsemble_BestOfFamily_3_AutoML_2_20250530_224731,0.180525,0.032589,0.139413,NaN,0.032589
7,StackedEnsemble_BestOfFamily_2_AutoML_2_20250530_224731,0.180640,0.032631,0.139467,NaN,0.032631
8,GBM_grid_1_AutoML_2_20250530_224731_model_22,0.180674,0.032643,0.139576,NaN,0.032643
9,StackedEnsemble_BestOfFamily_1_AutoML_2_20250530_224731,0.180679,0.032645,0.139495,NaN,0.032645


In [ ]:
#best_model = h2o.get_model(leaderboard_df.loc[0, 'model_id'])
#best_model.params
#aml.leader #gives the best model
# Plot variable importance
#best_model.varimp_plot()

key,value
Stacking strategy,cross_validation
Number of base models (used / total),13/38
# GBM base models (used / total),10/31
# DeepLearning base models (used / total),2/4
# DRF base models (used / total),1/2
# GLM base models (used / total),0/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [81]:
from autogluon.tabular import TabularPredictor

# Combine features and target into one DataFrame
train_df = combined_train
test_df = harps

# Create predictor
predictor = TabularPredictor(label='fe_h', eval_metric='mae').fit(train_df, time_limit=300)
# Predict on test set
y_pred = predictor.predict(test_df)

# Evaluate
from sklearn.metrics import mean_absolute_error
print("AutoGluon MAE:", mean_absolute_error(harps['fe_h'], y_pred))


No path specified. Models will be saved in: "AutogluonModels\ag-20250530_215939"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.7
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          22
Memory Avail:       8.60 GB / 31.74 GB (27.1%)
Disk Space Avail:   624.15 GB / 928.60 GB (67.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'       

AutoGluon MAE: 0.17449793895016213


In [82]:
predictor.leaderboard(silent=False) # shows performance of the models

                  model  score_val          eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2  -0.136643  mean_absolute_error       0.368793  211.080400                0.000973           0.080374            2       True         11
1        NeuralNetTorch  -0.137368  mean_absolute_error       0.013999  164.985607                0.013999         164.985607            1       True         10
2               XGBoost  -0.137463  mean_absolute_error       0.018244    1.033165                0.018244           1.033165            1       True          9
3              LightGBM  -0.137560  mean_absolute_error       0.007627    0.936078                0.007627           0.936078            1       True          4
4              CatBoost  -0.137725  mean_absolute_error       0.003000   18.182868                0.003000          18.182868            1       True          6
5       NeuralNetFastAI  -0.137791

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.136643,mean_absolute_error,0.368793,211.080400,0.000973,0.080374,2,True,11
1,NeuralNetTorch,-0.137368,mean_absolute_error,0.013999,164.985607,0.013999,164.985607,1,True,10
2,XGBoost,-0.137463,mean_absolute_error,0.018244,1.033165,0.018244,1.033165,1,True,9
3,LightGBM,-0.137560,mean_absolute_error,0.007627,0.936078,0.007627,0.936078,1,True,4
4,CatBoost,-0.137725,mean_absolute_error,0.003000,18.182868,0.003000,18.182868,1,True,6
5,NeuralNetFastAI,-0.137791,mean_absolute_error,0.089304,62.793269,0.089304,62.793269,1,True,8
6,LightGBMXT,-0.137839,mean_absolute_error,0.011999,1.714610,0.011999,1.714610,1,True,3
7,ExtraTreesMSE,-0.139867,mean_absolute_error,0.078656,36.682459,0.078656,36.682459,1,True,7
8,RandomForestMSE,-0.139898,mean_absolute_error,0.224910,8.241899,0.224910,8.241899,1,True,5
9,KNeighborsDist,-0.149624,mean_absolute_error,0.032012,0.056896,0.032012,0.056896,1,True,2


In [86]:
leaderboard_df = predictor.leaderboard(silent=True)
best_model = leaderboard_df.iloc[0]['model']
print("Best model:", best_model)
y_pred = predictor.predict(test_df, model=best_model)
print("MAE:", mean_absolute_error(test_df['fe_h'], y_pred))
performance = predictor.evaluate_predictions(
    y_true=test_df['fe_h'],
    y_pred=predictor.predict(test_df, model='LightGBM'),
    auxiliary_metrics=True
)
print(performance)

Best model: WeightedEnsemble_L2
MAE: 0.17449793895016213
{'mean_absolute_error': -0.17414989922259208, 'root_mean_squared_error': -0.22226393815269127, 'mean_squared_error': -0.049401258203143374, 'r2': 0.26573491428557405, 'pearsonr': 0.5743482895074574, 'median_absolute_error': -0.14266489297151563}
